In [1863]:
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
import pandas as pd
import numpy as np
import subprocess
import os
from datetime import datetime, timedelta, time, date
import datetime
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from textblob import TextBlob
from nltk.sentiment import SentimentIntensityAnalyzer
import speech_recognition as sr
from pydub import AudioSegment
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import nltk
from wordcloud import WordCloud
from collections import Counter
import re
import unicodedata

In [1864]:
mes = 'mayo'

In [1865]:
df = pd.read_csv(
    f'C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/Automatización/{mes}_notas.csv',
    sep=';',
    encoding='utf-8',
    engine='python'
)

df.head(2)

,Fecha,Concesión,Concesionario de Operación,Tipo de Servicio,Id de línea,Nombre Línea,N° Vehículo,Código Conductor,Nombre Conductor,ID Nota,Tipo Nota,Subtipo Nota,Subsubtipo Nota,Observaciones,Creado Por,Modificado Por
0,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993891,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 123251/ 01-05-2026/ 0...,ESTEBAN MEDINA CAMILO,ESTEBAN MEDINA CAMILO
1,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993893,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 123260/ 01-05-2026/ 0...,ESTEBAN MEDINA CAMILO,ESTEBAN MEDINA CAMILO


In [1866]:
tipos_validos = [
    'INTERRUPCION DE SERVICIO ZONAL'
]

df = df[
    df['Tipo Nota'].isin(tipos_validos)
]

print('\nCantidad después filtro Tipo Nota:')
print(len(df))

print('\nCantidad original registros:')
print(len(df))


# LIMPIEZA TEXTO


def limpiar_texto(texto):

    if pd.isna(texto):
        return ''

    texto = str(texto)

    # quitar saltos linea
    texto = texto.replace('\n', ' ')
    texto = texto.replace('\r', ' ')
    texto = texto.replace('\t', ' ')

    # minusculas
    texto = texto.lower()

    # normalizar unicode
    texto = unicodedata.normalize(
        'NFKD',
        texto
    )

    # quitar tildes
    texto = texto.encode(
        'ascii',
        'ignore'
    ).decode(
        'utf-8',
        errors='ignore'
    )

    # quitar espacios múltiples
    texto = re.sub(r'\s+', ' ', texto)

    # quitar espacios extremos
    texto = texto.strip()

    return texto


# APLICAR LIMPIEZA


df['obs_limpia'] = df['Observaciones'].apply(
    limpiar_texto
)


# FILTRAR EMPALME Y FASE


df = df[
    ~df['obs_limpia'].str.contains(
        'empalme|fase',
        na=False
    )
]


# METRICAS ESTRUCTURALES



# Cantidad palabras


df['cant_palabras'] = df['obs_limpia'].apply(
    lambda x: len(str(x).split())
)


# Cantidad "/"


df['cant_slash'] = df['Observaciones'].apply(
    lambda x: str(x).count('/')
)

# Cantidad números


df['cant_numeros'] = df['obs_limpia'].apply(
    lambda x: len(
        re.findall(
            r'\d+',
            str(x)
        )
    )
)


# Tiene fecha


df['tiene_fecha'] = df['obs_limpia'].apply(
    lambda x: bool(
        re.search(
            r'\d{2}[-/]\d{2}[-/]\d{4}',
            str(x)
        )
    )
)


# Tiene hora


df['tiene_hora'] = df['obs_limpia'].apply(
    lambda x: bool(
        re.search(
            r'\d{2}:\d{2}',
            str(x)
        )
    )
)


# Longitud texto


df['longitud_texto'] = df['obs_limpia'].apply(
    lambda x: len(str(x))
)


# DIVIDIR SEGMENTOS


def dividir_segmentos(texto):

    segmentos = str(texto).split('/')

    segmentos = [
        s.strip()
        for s in segmentos
        if s.strip() != ''
    ]

    return segmentos


# APLICAR SEGMENTOS


df['segmentos'] = df['Observaciones'].apply(
    dividir_segmentos
)


# CONTAR SEGMENTOS


df['cantidad_segmentos'] = df['segmentos'].apply(
    len
)


# CLASIFICACION


def clasificar_estructura(row):

    score = 0

    # muchos "/"
    if row['cant_slash'] >= 5:
        score += 1

    # muchas palabras
    if row['cant_palabras'] >= 15:
        score += 1

    # varios números
    if row['cant_numeros'] >= 3:
        score += 1

    # tiene fecha
    if row['tiene_fecha']:
        score += 1

    # tiene hora
    if row['tiene_hora']:
        score += 1

    # clasificación final
    if score >= 4:
        return 'MUY_PROBABLE_DESVIO'

    elif score >= 2:
        return 'POSIBLE_DESVIO'

    else:
        return 'OTRO'

# APLICAR CLASIFICACION


df['clasificacion'] = df.apply(
    clasificar_estructura,
    axis=1
)


# LIMPIAR COLUMNAS TEXTO
# EVITA FILAS EN BLANCO EN EXCEL


columnas_texto = [
    'Observaciones',
    'obs_limpia'
]

for col in columnas_texto:

    df[col] = (
        df[col]
        .astype(str)
        .str.replace('\n', ' ', regex=False)
        .str.replace('\r', ' ', regex=False)
        .str.replace('\t', ' ', regex=False)
        .str.replace(';', ',', regex=False)
        .str.replace('"', '', regex=False)
        .str.strip()
    )


# RESULTADOS


print('\nClasificación:\n')

print(
    df['clasificacion'].value_counts()
)

# EJEMPLOS MUY PROBABLE DESVIO

print('\nEjemplos MUY_PROBABLE_DESVIO:\n')

print(
    df[
        df['clasificacion']
        == 'MUY_PROBABLE_DESVIO'
    ][[
        'Observaciones',
        'cant_palabras',
        'cant_slash',
        'cant_numeros',
        'cantidad_segmentos'
    ]].head(20)
)

# SEGMENTOS FRECUENTES

segmentos = df['segmentos'].explode()

print('\nSegmentos más frecuentes:\n')

print(
    segmentos.value_counts().head(50)
)

# EXPORTAR CSV LIMPIO

ruta_exportacion = (
    f'C:/Users/Jonny Villareal/'
    f'OneDrive - Gmovil SAS/'
    f'Escritorio/Jonny/'
    f'Control 2026/'
    f'Automatización/'
    f'{mes}_desvios_extraidos.csv'
)

df.to_csv(
    ruta_exportacion,
    sep=';',
    index=False,
    encoding='utf-8-sig'
)




Cantidad después filtro Tipo Nota:
1536

Cantidad original registros:
1536

Clasificación:

clasificacion
MUY_PROBABLE_DESVIO    1533
POSIBLE_DESVIO            2
Name: count, dtype: int64

Ejemplos MUY_PROBABLE_DESVIO:

                                        Observaciones  cant_palabras  \
0   Autorización desvío/ ID: 123251/ 01-05-2026/ 0...             85   
1   Autorización desvío/ ID: 123260/ 01-05-2026/ 0...             79   
2   Autorización desvío/ ID: 123262/ 01-05-2026/ 0...             76   
3   Autorización desvío/ ID: 123513/ 01-05-2026/ 0...             72   
4   Autorización desvío/ ID: 123516/ 01-05-2026/ 0...             75   
5   Autorización desvío/ ID: 123358/ 01-05-2026/ 0...            121   
6   Autorización desvío/ ID: 123540/ 01-05-2026/ 0...             79   
7   Autorización desvío/ ID: 122172/ 01-05-2026/ 0...             63   
8   Autorización desvío/ ID: 123223/ 01-05-2026/ 0...             69   
9   Autorización desvío/ ID: 123553/ 01-05-2026/ 0...      

In [1867]:
ruta_exportacion = f'C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/Automatización/{mes}_desvios_extraidos.csv'

df = pd.read_csv(
    ruta_exportacion,
    sep=';',
    encoding='utf-8-sig'
)

print(df.shape)
df.head(2)

(1535, 26)


,Fecha,Concesión,Concesionario de Operación,Tipo de Servicio,Id de línea,Nombre Línea,N° Vehículo,Código Conductor,Nombre Conductor,ID Nota,...,obs_limpia,cant_palabras,cant_slash,cant_numeros,tiene_fecha,tiene_hora,longitud_texto,segmentos,cantidad_segmentos,clasificacion
0,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993891,...,autorizacion desvio/ id: 123251/ 01-05-2026/ 0...,85,13,19,True,True,524,"['Autorización desvío', 'ID: 123251', '01-05-2...",14,MUY_PROBABLE_DESVIO
1,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993893,...,autorizacion desvio/ id: 123260/ 01-05-2026/ 0...,79,13,19,True,True,475,"['Autorización desvío', 'ID: 123260', '01-05-2...",14,MUY_PROBABLE_DESVIO


In [1868]:
df = df[df['clasificacion'] == 'MUY_PROBABLE_DESVIO'].copy()

In [1869]:
def extraer_campos(texto):

    if pd.isna(texto):
        return pd.Series()

    data = {
        "ID": None,
        "Fecha": None,
        "Hora": None,
        "Ruta": None,
        "Motivo": None,
        "Direccion": None,
        "Sentido": None,
        "Desde": None,
        "Hasta": None,
        "Descripcion": None
    }

    partes = [p.strip() for p in str(texto).split('/')]

    for p in partes:

        # ID
        if p.startswith("ID"):
            data["ID"] = re.search(r"\d+", p)
            data["ID"] = data["ID"].group() if data["ID"] else None

        # Fecha
        elif re.search(r"\d{2}-\d{2}-\d{4}", p):
            data["FechaR"] = re.search(r"\d{2}-\d{2}-\d{4}", p).group()

        # Hora
        elif re.search(r"\d{2}:\d{2}", p):
            data["Hora"] = re.search(r"\d{2}:\d{2}", p).group()

        # Ruta
        elif "ruta" in p.lower():
            data["Ruta"] = p

        # Motivo
        elif "motivo" in p.lower():
            data["Motivo"] = p.split(":", 1)[-1].strip()

        # Dirección
        elif "direccion" in p.lower():
            data["Direccion"] = p.split(":", 1)[-1].strip()

        # Sentido
        elif "sentido" in p.lower():
            data["Sentido"] = p.split(":", 1)[-1].strip()

        # Desde
        elif "desde" in p.lower():
            m = re.search(r"(\w+\d+)", p)
            data["Desde"] = m.group() if m else None

        # Hasta
        elif "hasta" in p.lower():
            m = re.search(r"(\w+\d+)", p)
            data["Hasta"] = m.group() if m else None

        # Descripción
        elif "desvio" in p.lower():
            data["Descripcion"] = p.split(":", 1)[-1].strip()

    return pd.Series(data)

In [1870]:
df_campos = df['Observaciones'].apply(extraer_campos)

df_final = pd.concat([df, df_campos], axis=1)

In [1871]:
# ELIMINAR REGISTROS SIN ID

df_final = df_final[
    df_final['ID'].notna()
]

# También quitar vacíos reales
df_final = df_final[
    df_final['ID'].astype(str).str.strip() != ''
]

print('\nCantidad después quitar ID vacíos:')
print(len(df_final))


Cantidad después quitar ID vacíos:
1488


In [1872]:

# ELIMINAR REGISTROS SIN FECHA

df_final = df_final[
    df_final['FechaR'].notna()
]

# quitar vacíos reales
df_final = df_final[
    df_final['FechaR'].astype(str).str.strip() != ''
]

print('\nCantidad después quitar Fecha vacía:')
print(len(df_final))


Cantidad después quitar Fecha vacía:
1477


In [1873]:

# COMPLETAR MOTIVO DESDE OBSERVACIONES


def completar_motivo(row):

    motivo = str(row['Motivo']).strip().lower()
    obs = str(row['Observaciones']).lower()

    # Si ya tiene motivo válido
    if motivo not in ['', 'nan', 'none']:
        return row['Motivo']

    
    # BUSQUEDA PALABRAS CLAVE
   

    if 'obra' in obs:
        return 'Obras en la vía'
    
    if 'bloqueo' in obs:
        return 'Cierre vial'
    
    if 'cierre' in obs:
        return 'Manifestación'
    
    elif 'manifestaciónes' in obs:
        return 'Manifestación'
    
    elif 'manifestacion' in obs:
        return 'Manifestación'
        
    elif 'manifestación' in obs:
        return 'Manifestación'

    elif 'accidente' in obs:
        return 'Accidente vial'

    elif 'malla vial' in obs:
        return 'Mal estado de la malla vial'

    elif 'evento' in obs:
        return 'Evento'

    elif 'inundacion' in obs:
        return 'Inundación'

    elif 'hundimiento' in obs:
        return 'Hundimiento vial'

    elif 'movilizacion' in obs:
        return 'Movilización social'

    elif 'cierre vial' in obs:
        return 'Cierre vial'

    elif 'congestion' in obs:
        return 'Congestión vehicular'
    
    elif 'carrera' in obs:
        return 'Evento deportivo'
    
    elif 'ciclovia' in obs:
        return 'Evento deportivo'

    elif 'cerrada' in obs:
        return 'Cierre vial'
    
    elif 'choque' in obs:
        return 'Accidente'
    
    elif 'varado' in obs:
        return 'Falla mécanica'
    
    elif 'congestión' in obs:
        return 'Congestión vehicular'
    
    elif 'zonal bloqueado' in obs:
        return 'Falla mécanica'
    
    # SIN DETECCION
   

    return None

In [1874]:
df_final['Motivo'] = df_final.apply(
    completar_motivo,
    axis=1
)

In [1875]:

# HOMOLOGACION MOTIVOS


homologacion = {

    # MALLA VIAL
    

    'Mal estado de la Malla vial': 'Malla vial',
    'Daño Malla Vial': 'Malla vial',
    'Daño Malla Vial ( ciclovía )': 'Malla vial',
    'Mal estado maya Vial': 'Malla vial',
    'Mal estado malla Vial': 'Malla vial',

    
    # OBRAS
    

    'Obras en la via': 'Obras',
    'Obras en la vía': 'Obras',
    'obras': 'Obras',
    'obras mantenimiento vial': 'Obras',
    'Obras primera línea Metro Bogotá': 'Obras',
    'Obras primera línea Metro Bogotá ( ciclovía)': 'Obras',
    'obras de condensa colocando postes de energía': 'Obras',
    'Obras en la vía (Desvío Ciclovía)': 'Obras',
    'Obras en la via (ciclovía )': 'Obras',
    'obras en vía, cambio de sentido': 'Obras',

    
    # MANIFESTACION
    

    'Manifestación': 'Manifestación',

    
    # EVENTO
    

    'Evento': 'Evento',
    'Evento deportivo': 'Evento',

   
    # CIERRE VIAL
    

    'Cierre vial': 'Cierre vial',
    'Cierre Gaitán cortes': 'Cierre vial',
    'Cierre por movilidad  en la Av Gaitán cortes': 'Cierre vial',
    'cerrado por policia vigilancia': 'Cierre vial',
    'policia cerrada la via': 'Cierre vial',

   
    # ACCIDENTE
   

    'Accidente vial': 'Accidente',
    'accidente con fatalidad': 'Accidente',

    
    # RESTRICCION
    

    'Paso reducido': 'Restricción vial',
    'Cambio de sentido': 'Restricción vial',

   
    # OPERACIONAL
    

    'Retoma trazado original': 'Operacional',

   
    # CONGESTION
   

    'Congestión vehicular': 'Congestión',

   
    # FALLA MECANICA
   

    'bus zonal Z25-2081 varado': 'Falla mecánica',
    'Falla mécanica': 'Falla mecánica'

}


# APLICAR HOMOLOGACION

df_final['Motivo_R'] = df_final['Motivo']


df_final['Motivo'] = (
    df_final['Motivo']
    .replace(homologacion)
)


# PRIMERA LETRA MAYUSCULA


df_final['Motivo'] = (
    df_final['Motivo']
    .astype(str)
    .str.strip()
    .str.title()
)


# RESULTADOS


print('\nMotivos homologados:\n')

print(
    df_final['Motivo']
    .value_counts()
)


Motivos homologados:

Motivo
Obras                       555
Malla Vial                  214
Restricción Vial            208
Accidente                   205
Manifestación               112
Cierre Vial                  80
Evento                       72
Operacional                  16
Motivo                        7
Falla Mecánica                4
Congestión                    2
Obras En La Via               1
Obras Mantenimiento Vial      1
Name: count, dtype: int64


In [1876]:

# EXTRAER SENTIDO ROBUSTO


import re
import pandas as pd

def extraer_sentido(texto):

    if pd.isna(texto):
        return None

    texto = str(texto).lower()

    
    # LIMPIEZA GENERAL
  

    texto = re.sub(r'\s+', ' ', texto)

    # quitar ;
    texto = texto.replace(';', ' ')

    # quitar :
    texto = texto.replace(':', ' ')

    # quitar dobles espacios
    texto = texto.strip()

   
    # PATRONES FLEXIBLES
    

    patrones = {

        'Oriente - Occidente': [
            r'oriente\s*-?\s*occidente'
        ],

        'Occidente - Oriente': [
            r'occidente\s*-?\s*oriente'
        ],

        'Norte - Sur': [
            r'norte\s*-?\s*sur'
        ],

        'Sur - Norte': [
            r'sur\s*-?\s*norte'
        ]
    }

 
    # BUSCAR


    for sentido, lista in patrones.items():

        for patron in lista:

            if re.search(patron, texto):
                return sentido

    return None

In [1877]:
df_final['Sentido'] = df_final['Observaciones'].apply(
    extraer_sentido
)

In [1878]:

# EXTRAER DESDE Y HASTA


import pandas as pd
import re

def extraer_desde_hasta(texto):

    if pd.isna(texto):

        return pd.Series({
            'Desde': None,
            'Hasta': None
        })

    texto = str(texto)


    # BUSCAR CÓDIGOS:
   

    codigos = re.findall(
        r'\d{3}[A-Za-zª]\d{2}',
        texto
    )

    resultado = {
        'Desde': None,
        'Hasta': None
    }


    # DESDE


    if len(codigos) >= 1:

        resultado['Desde'] = codigos[0]

    # HASTA


    if len(codigos) >= 2:

        resultado['Hasta'] = codigos[1]

    return pd.Series(resultado)


# ELIMINAR COLUMNAS SI EXISTEN


df_final = df_final.drop(
    columns=['Desde', 'Hasta'],
    errors='ignore'
)


# APLICAR


df_desde_hasta = df_final['Observaciones'].apply(
    extraer_desde_hasta
)


# UNIR


df_final = pd.concat(
    [df_final, df_desde_hasta],
    axis=1
)

# RESULTADO


print(
    df_final[
        ['Desde', 'Hasta']
    ].head(50)
)

     Desde   Hasta
0   052A08  429A08
1   088A13  175A13
2   173A13  089A13
3   315A00  952V00
4   050A00  227A00
5   251B01  168B01
6   426A01  146B01
7   050A00  227B00
8   280A00  002A00
9   152A00  230A00
10  330A05    None
11  355A00  300A00
12  405A05  225A05
13  099D09  510B09
14  052A08  114A08
15  162C08  114B09
16  098A09  154C08
17  163A08  621B09
18  330A08  541A08
19  112C05  185B05
20  504A06  159A06
21  314A00  952V00
22  192A00  280A00
23  355B00  913A00
24  441A01  064A01
25  441A06  159A06
26  263A13  266A13
27  094B00  315A01
28  441A01  064A01
29  050A00  228B00
30  441A06  159A06
31  355B00  913A00
32  799A00  315A01
33  192A00  230A00
34  251B01  168B01
35  251B01  168B01
36  638B08  035A08
37  110A00  251B00
38  191A00  519A00
39  110A00  251B00
42  168A05  114A06
43  158A06  062A05
44  258A00  337A13
46  541A13  067B00
47  403A05  114A06
48  403A05  114A06
49  120B07  395A06
50  120B07  395A06
51  234A00  352B00
52  234A00  352B00


In [1879]:

df_final['Desde'] = (
    df_final['Desde']
    .astype(str)
    .str.strip()
)

df_final['Hasta'] = (
    df_final['Hasta']
    .astype(str)
    .str.strip()
)


df_final = df_final[
    (df_final['Desde'].notna()) &
    (df_final['Hasta'].notna()) &
    (df_final['Desde'] != '') &
    (df_final['Hasta'] != '') &
    (df_final['Desde'].str.lower() != 'none') &
    (df_final['Hasta'].str.lower() != 'none') &
    (df_final['Desde'].str.lower() != 'nan') &
    (df_final['Hasta'].str.lower() != 'nan')
]


print('\nCantidad final:')
print(len(df_final))


Cantidad final:
1465


In [1880]:
df_final.head(2)

,Fecha,Concesión,Concesionario de Operación,Tipo de Servicio,Id de línea,Nombre Línea,N° Vehículo,Código Conductor,Nombre Conductor,ID Nota,...,Hora,Ruta,Motivo,Direccion,Sentido,Descripcion,FechaR,Motivo_R,Desde,Hasta
0,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993891,...,03:00,Ruta 740,Malla Vial,None,Occidente - Oriente,None,01-05-2026,Mal estado de la Malla vial,052A08,429A08
1,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993893,...,03:00,Ruta 740,Restricción Vial,None,Norte - Sur,None,01-05-2026,Paso reducido,088A13,175A13


In [1881]:
def normalizar_codigo(texto):

    if pd.isna(texto):
        return None

    texto = str(texto)

    # 🔥 reemplazos directos de caracteres problemáticos
    texto = texto.replace("ª", "A")
    texto = texto.replace("º", "O")

    # 🔥 opcional: asegurar mayúscula
    texto = texto.upper()

    return texto

df_final["Desde"] = df_final["Desde"].apply(normalizar_codigo)
df_final["Hasta"] = df_final["Hasta"].apply(normalizar_codigo)

df_final["Desde"] = df_final["Desde"].astype(str).str.strip().str.upper()
df_final["Hasta"] = df_final["Hasta"].astype(str).str.strip().str.upper()

Paradas

In [1882]:
#importar paradas

paradas = pd.read_csv('C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/Paraderos_Zonales_del_SITP.csv')

paradas.head(3)

,X,Y,objectid,cenefa,zona_sitp,nombre,via,direccion_bandera,localidad,longitud,latitud,consecutivo_zona,tipo_m_s,consola,panel,audio,zonas_nuevas,globalid,shape
0,1.001502e+06,1.010205e+06,1,001A00,00,C.C. Iserra 100,AC 100,AC 100 - KR 54,Barrios Unidos,-74.063971,4.688481,001,M,AC 100 - KR 54 (001A00),AC 100 - KR 54,Avenida Calle 100 Carrera 54,C,{1C0DBC4E-15BC-4BBE-BC16-5628077DBE2E},NaN
1,1.003505e+06,1.009719e+06,2,001A01,01,Br. Rincón del Chicó,AC 100,AC 100 - KR 13,Usaquén,-74.045914,4.684091,001,M,AC 100 - KR 13 (001A01),AC 100 - KR 13,Avenida Calle 100 Carrera 13,B,{60A22A44-AD56-4DF4-A3E6-6830B9FB0792},NaN
2,1.001238e+06,1.018098e+06,3,001A02,02,Gimnasio Iragua,AV. Boyacá,AV. Boyacá - AC 170,Suba,-74.066350,4.759867,001,S,AV. Boyacá - AC 170 (001A02),AV. Boyacá - AC 170,Avenida Boyacá Avenida Calle 170,C,{97155274-E4D5-45A4-891F-2DCA19FF10D9},NaN


In [1883]:
paradas["cenefa"] = paradas["cenefa"].astype(str).str.strip().str.upper()

In [1884]:
#Cruzar con datos de nodo y Numero_parada

def calcular_turno(parada):
    
    filtro = (
        (paradas['cenefa'] == parada)
    )
    
    # Verificar si hay algún resultado después de aplicar el filtro
    if not paradas.loc[filtro].empty:
        # Obtener el primer valor
        paradas1 = paradas.loc[filtro, 'latitud'].iloc[0]
        return paradas1 if not pd.isna(paradas1) else None  
    
    return None  # Devolver None si no hay resultados

# Aplica la función a las columnas correspondientes
df_final['latitud_desde'] = df_final.apply(
    lambda row: calcular_turno(
        row['Desde']
    ),
    axis=1
)

df_final.head()

,Fecha,Concesión,Concesionario de Operación,Tipo de Servicio,Id de línea,Nombre Línea,N° Vehículo,Código Conductor,Nombre Conductor,ID Nota,...,Ruta,Motivo,Direccion,Sentido,Descripcion,FechaR,Motivo_R,Desde,Hasta,latitud_desde
0,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993891,...,Ruta 740,Malla Vial,None,Occidente - Oriente,None,01-05-2026,Mal estado de la Malla vial,052A08,429A08,4.638358
1,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993893,...,Ruta 740,Restricción Vial,None,Norte - Sur,None,01-05-2026,Paso reducido,088A13,175A13,4.546156
2,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993895,...,Ruta 740,Restricción Vial,None,Sur - Norte,None,01-05-2026,Paso reducido,173A13,089A13,4.537105
3,1/05/2026 2:13,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10196,E25,0,0,,993900,...,Descripción de desvío: En calle 72 con carrera...,Restricción Vial,None,Oriente - Occidente,None,01-05-2026,Cambio de sentido,315A00,952V00,4.656979
4,1/05/2026 2:13,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10196,E25,0,0,,993901,...,Descripción de desvío: En calle 72 con carrera...,Restricción Vial,None,Occidente - Oriente,None,01-05-2026,Cambio de sentido,050A00,227A00,4.662240


In [1885]:
#Cruzar con datos de nodo y Numero_parada

def calcular_turno(parada):
    
    filtro = (
        (paradas['cenefa'] == parada)
    )
    
    # Verificar si hay algún resultado después de aplicar el filtro
    if not paradas.loc[filtro].empty:
        # Obtener el primer valor
        paradas1 = paradas.loc[filtro, 'longitud'].iloc[0]
        return paradas1 if not pd.isna(paradas1) else None  
    
    return None  # Devolver None si no hay resultados

# Aplica la función a las columnas correspondientes
df_final['longitud_desde'] = df_final.apply(
    lambda row: calcular_turno(
        row['Desde']
    ),
    axis=1
)

df_final.head()

,Fecha,Concesión,Concesionario de Operación,Tipo de Servicio,Id de línea,Nombre Línea,N° Vehículo,Código Conductor,Nombre Conductor,ID Nota,...,Motivo,Direccion,Sentido,Descripcion,FechaR,Motivo_R,Desde,Hasta,latitud_desde,longitud_desde
0,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993891,...,Malla Vial,None,Occidente - Oriente,None,01-05-2026,Mal estado de la Malla vial,052A08,429A08,4.638358,-74.155502
1,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993893,...,Restricción Vial,None,Norte - Sur,None,01-05-2026,Paso reducido,088A13,175A13,4.546156,-74.091717
2,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993895,...,Restricción Vial,None,Sur - Norte,None,01-05-2026,Paso reducido,173A13,089A13,4.537105,-74.086317
3,1/05/2026 2:13,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10196,E25,0,0,,993900,...,Restricción Vial,None,Oriente - Occidente,None,01-05-2026,Cambio de sentido,315A00,952V00,4.656979,-74.058146
4,1/05/2026 2:13,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10196,E25,0,0,,993901,...,Restricción Vial,None,Occidente - Oriente,None,01-05-2026,Cambio de sentido,050A00,227A00,4.662240,-74.066179


In [1886]:
#Cruzar con datos de nodo y Numero_parada

def calcular_turno(parada):
    
    filtro = (
        (paradas['cenefa'] == parada)
    )
    
    # Verificar si hay algún resultado después de aplicar el filtro
    if not paradas.loc[filtro].empty:
        # Obtener el primer valor
        paradas1 = paradas.loc[filtro, 'latitud'].iloc[0]
        return paradas1 if not pd.isna(paradas1) else None  
    
    return None  # Devolver None si no hay resultados

# Aplica la función a las columnas correspondientes
df_final['latitud_hasta'] = df_final.apply(
    lambda row: calcular_turno(
        row['Hasta']
    ),
    axis=1
)

df_final.head()

,Fecha,Concesión,Concesionario de Operación,Tipo de Servicio,Id de línea,Nombre Línea,N° Vehículo,Código Conductor,Nombre Conductor,ID Nota,...,Direccion,Sentido,Descripcion,FechaR,Motivo_R,Desde,Hasta,latitud_desde,longitud_desde,latitud_hasta
0,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993891,...,None,Occidente - Oriente,None,01-05-2026,Mal estado de la Malla vial,052A08,429A08,4.638358,-74.155502,4.630447
1,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993893,...,None,Norte - Sur,None,01-05-2026,Paso reducido,088A13,175A13,4.546156,-74.091717,4.543940
2,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993895,...,None,Sur - Norte,None,01-05-2026,Paso reducido,173A13,089A13,4.537105,-74.086317,4.546430
3,1/05/2026 2:13,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10196,E25,0,0,,993900,...,None,Oriente - Occidente,None,01-05-2026,Cambio de sentido,315A00,952V00,4.656979,-74.058146,NaN
4,1/05/2026 2:13,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10196,E25,0,0,,993901,...,None,Occidente - Oriente,None,01-05-2026,Cambio de sentido,050A00,227A00,4.662240,-74.066179,4.656000


In [1887]:
#Cruzar con datos de nodo y Numero_parada

def calcular_turno(parada):
    
    filtro = (
        (paradas['cenefa'] == parada)
    )
    
    # Verificar si hay algún resultado después de aplicar el filtro
    if not paradas.loc[filtro].empty:
        # Obtener el primer valor
        paradas1 = paradas.loc[filtro, 'longitud'].iloc[0]
        return paradas1 if not pd.isna(paradas1) else None  
    
    return None  # Devolver None si no hay resultados

# Aplica la función a las columnas correspondientes
df_final['longitud_hasta'] = df_final.apply(
    lambda row: calcular_turno(
        row['Hasta']
    ),
    axis=1
)

df_final.head()

,Fecha,Concesión,Concesionario de Operación,Tipo de Servicio,Id de línea,Nombre Línea,N° Vehículo,Código Conductor,Nombre Conductor,ID Nota,...,Sentido,Descripcion,FechaR,Motivo_R,Desde,Hasta,latitud_desde,longitud_desde,latitud_hasta,longitud_hasta
0,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993891,...,Occidente - Oriente,None,01-05-2026,Mal estado de la Malla vial,052A08,429A08,4.638358,-74.155502,4.630447,-74.153543
1,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993893,...,Norte - Sur,None,01-05-2026,Paso reducido,088A13,175A13,4.546156,-74.091717,4.543940,-74.091351
2,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993895,...,Sur - Norte,None,01-05-2026,Paso reducido,173A13,089A13,4.537105,-74.086317,4.546430,-74.091850
3,1/05/2026 2:13,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10196,E25,0,0,,993900,...,Oriente - Occidente,None,01-05-2026,Cambio de sentido,315A00,952V00,4.656979,-74.058146,NaN,NaN
4,1/05/2026 2:13,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10196,E25,0,0,,993901,...,Occidente - Oriente,None,01-05-2026,Cambio de sentido,050A00,227A00,4.662240,-74.066179,4.656000,-74.059659


In [1888]:
#Cruzar con datos de nodo y Numero_parada

def calcular_turno(parada):
    
    filtro = (
        (paradas['cenefa'] == parada)
    )
    
    # Verificar si hay algún resultado después de aplicar el filtro
    if not paradas.loc[filtro].empty:
        # Obtener el primer valor
        paradas1 = paradas.loc[filtro, 'direccion_bandera'].iloc[0]
        return paradas1 if not pd.isna(paradas1) else None  
    
    return None  # Devolver None si no hay resultados

# Aplica la función a las columnas correspondientes
df_final['direccion_desde'] = df_final.apply(
    lambda row: calcular_turno(
        row['Desde']
    ),
    axis=1
)

df_final.head()

,Fecha,Concesión,Concesionario de Operación,Tipo de Servicio,Id de línea,Nombre Línea,N° Vehículo,Código Conductor,Nombre Conductor,ID Nota,...,Descripcion,FechaR,Motivo_R,Desde,Hasta,latitud_desde,longitud_desde,latitud_hasta,longitud_hasta,direccion_desde
0,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993891,...,None,01-05-2026,Mal estado de la Malla vial,052A08,429A08,4.638358,-74.155502,4.630447,-74.153543,AV. Américas - KR 82
1,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993893,...,None,01-05-2026,Paso reducido,088A13,175A13,4.546156,-74.091717,4.543940,-74.091351,DG 43A Sur - KR 6A E
2,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993895,...,None,01-05-2026,Paso reducido,173A13,089A13,4.537105,-74.086317,4.546430,-74.091850,DG 48 Sur - TV 13D Este
3,1/05/2026 2:13,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10196,E25,0,0,,993900,...,None,01-05-2026,Cambio de sentido,315A00,952V00,4.656979,-74.058146,NaN,NaN,AC 72 - KR 10
4,1/05/2026 2:13,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10196,E25,0,0,,993901,...,None,01-05-2026,Cambio de sentido,050A00,227A00,4.662240,-74.066179,4.656000,-74.059659,AC 72 - KR 22


In [1889]:
#Cruzar con datos de nodo y Numero_parada

def calcular_turno(parada):
    
    filtro = (
        (paradas['cenefa'] == parada)
    )
    
    # Verificar si hay algún resultado después de aplicar el filtro
    if not paradas.loc[filtro].empty:
        # Obtener el primer valor
        paradas1 = paradas.loc[filtro, 'direccion_bandera'].iloc[0]
        return paradas1 if not pd.isna(paradas1) else None  
    
    return None  # Devolver None si no hay resultados

# Aplica la función a las columnas correspondientes
df_final['direccion_hasta'] = df_final.apply(
    lambda row: calcular_turno(
        row['Hasta']
    ),
    axis=1
)

df_final.head()

,Fecha,Concesión,Concesionario de Operación,Tipo de Servicio,Id de línea,Nombre Línea,N° Vehículo,Código Conductor,Nombre Conductor,ID Nota,...,FechaR,Motivo_R,Desde,Hasta,latitud_desde,longitud_desde,latitud_hasta,longitud_hasta,direccion_desde,direccion_hasta
0,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993891,...,01-05-2026,Mal estado de la Malla vial,052A08,429A08,4.638358,-74.155502,4.630447,-74.153543,AV. Américas - KR 82,CL 26 Sur - KR 79F
1,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993893,...,01-05-2026,Paso reducido,088A13,175A13,4.546156,-74.091717,4.543940,-74.091351,DG 43A Sur - KR 6A E,KR 7A E - DG 45A Sur
2,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993895,...,01-05-2026,Paso reducido,173A13,089A13,4.537105,-74.086317,4.546430,-74.091850,DG 48 Sur - TV 13D Este,DG 43A Sur - KR 6A E
3,1/05/2026 2:13,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10196,E25,0,0,,993900,...,01-05-2026,Cambio de sentido,315A00,952V00,4.656979,-74.058146,NaN,NaN,AC 72 - KR 10,None
4,1/05/2026 2:13,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10196,E25,0,0,,993901,...,01-05-2026,Cambio de sentido,050A00,227A00,4.662240,-74.066179,4.656000,-74.059659,AC 72 - KR 22,AK 11 - CL 70A


In [1890]:
ruta_exportacion = (
    f'C:/Users/Jonny Villareal/'
    f'OneDrive - Gmovil SAS/'
    f'Escritorio/Jonny/'
    f'Control 2026/'
    f'Automatización/'
    f'{mes}_desvios_extraidos1.csv'
)

df_final.to_csv(
    ruta_exportacion,
    sep=';',
    index=False,
    encoding='utf-8-sig'
)